In [18]:
import pyspark
from pyspark.sql import SparkSession
import os
import sys
import pyarrow
from pyspark.sql.functions import current_timestamp


# === 1. Wykrywanie katalogu PySpark ===
# pyspark_path = os.path.dirname(pyspark.__file__)
# hadoop_home_path = pyspark_path  # PySpark ma wbudowane jars w tym katalogu

# === 2. Ustawienie zmiennych środowiskowych ===
os.environ["HADOOP_HOME"] = r"C:\hadoop"
# Dodaj katalog bin do PATH
os.environ["PATH"] += r"C:\hadoop\bin"

# Ustawienie Pythona dla drivera i workerów
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print(f"[INFO] HADOOP_HOME set to: {os.environ['HADOOP_HOME']}")
print(f"[INFO] PYSPARK_PYTHON set to: {os.environ['PYSPARK_PYTHON']}")
print(f"[INFO] PYSPARK_DRIVER_PYTHON set to: {os.environ['PYSPARK_DRIVER_PYTHON']}")

# os.environ["PYSPARK_PYTHON"] = sys.executable
# os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
print(f"Zainstalowana wersja PyArrow: {pyarrow.__version__}")


spark = SparkSession.builder \
    .appName("Test") \
    .config("spark.driver.maxResultSize", "2g") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.3") \
    .config("spark.driver.memory", "2g") \
    .master("local[*]") \
    .getOrCreate()
print(spark.version)

[INFO] HADOOP_HOME set to: C:\hadoop
[INFO] PYSPARK_PYTHON set to: c:\Users\Si3ma\Desktop\spark_simulation\spark_workers_Test\.venv\Scripts\python.exe
[INFO] PYSPARK_DRIVER_PYTHON set to: c:\Users\Si3ma\Desktop\spark_simulation\spark_workers_Test\.venv\Scripts\python.exe
Zainstalowana wersja PyArrow: 25.0.1
4.2.0


In [19]:
arrow_enabled = spark.conf.get("spark.sql.execution.arrow.pyspark.enabled", "false")
print(f"PyArrow włączony: {arrow_enabled}")
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
arrow_enabled = spark.conf.get("spark.sql.execution.arrow.pyspark.enabled", "false")
print(f"PyArrow włączony: {arrow_enabled}")

PyArrow włączony: true
PyArrow włączony: true


In [3]:
from opensky_api import OpenSkyApi, TokenManager
with OpenSkyApi(token_manager=TokenManager.from_json_file("credentials.json")) as api:
    states = api.get_states()
states_df = states

In [4]:
lista={}
for state in states.states:
    # lista[state.icao24]['icao24'] = state.icao24
    lista[state.icao24] = {}
    lista[state.icao24]['callsign'] = state.callsign
    lista[state.icao24]['origin_country'] = state.origin_country
    lista[state.icao24]['time_position'] = state.time_position
    lista[state.icao24]['last_contact'] = state.last_contact
    lista[state.icao24]['longitude'] = state.longitude
    lista[state.icao24]['latitude'] = state.latitude
    lista[state.icao24]['geo_altitude'] = state.geo_altitude
    lista[state.icao24]['on_ground'] = state.on_ground
    lista[state.icao24]['velocity'] = state.velocity
    lista[state.icao24]['true_track'] = state.true_track
    lista[state.icao24]['vertical_rate'] = state.vertical_rate
    lista[state.icao24]['sensors'] = state.sensors
    lista[state.icao24]['baro_altitude'] = state.baro_altitude
    lista[state.icao24]['squawk'] = state.squawk
    lista[state.icao24]['spi'] = state.spi
    lista[state.icao24]['position_source'] = state.position_source
    lista[state.icao24]['category'] = state.category

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, MapType
schema = StructType([
StructField("icao24", StringType(), True),
StructField("callsign",  StringType(), True),
StructField("origin_country",  StringType(), True),
StructField("time_position",  StringType(), True),
StructField("last_contact",  StringType(), True),
StructField("longitude",  StringType(), True),
StructField("latitude",  StringType(), True),
StructField("geo_altitude",  StringType(), True),
StructField("on_ground",  StringType(), True),
StructField("velocity",  StringType(), True),
StructField("true_track",  StringType(), True),
StructField("vertical_rate",  StringType(), True),
StructField("sensors",  StringType(), True),
StructField("baro_altitude",  StringType(), True),
StructField("squawk",  StringType(), True),
StructField("spi",  StringType(), True),
StructField("position_source",  StringType(), True),
StructField("category",  StringType(), True)
])


In [6]:
aircraft_df=[]
for aircraft in lista:
    aircraft_df.extend(
    [(
        aircraft,
        lista[aircraft]['callsign'],
        lista[aircraft]['origin_country'], 
        lista[aircraft]['time_position'],
        lista[aircraft]['last_contact'],
        lista[aircraft]['longitude'],
        lista[aircraft]['latitude'],
        lista[aircraft]['geo_altitude'],
        lista[aircraft]['on_ground'],
        lista[aircraft]['velocity'],
        lista[aircraft]['true_track'],
        lista[aircraft]['vertical_rate'],
        lista[aircraft]['sensors'],
        lista[aircraft]['baro_altitude'],
        lista[aircraft]['squawk'],
        lista[aircraft]['spi'],
        lista[aircraft]['position_source'],
        lista[aircraft]['category']
        )])
    
    print(f"ICAO24: {aircraft}")
    print(lista[aircraft]['callsign'])
    print(lista[aircraft]['origin_country'])
    print(lista[aircraft]['time_position'])
    print(lista[aircraft]['last_contact'])
    print(lista[aircraft]['longitude'])
    print(lista[aircraft]['latitude'])
    print(lista[aircraft]['geo_altitude'])
    print(lista[aircraft]['on_ground'])
    print(lista[aircraft]['velocity'])
    print(lista[aircraft]['true_track'])
    print(lista[aircraft]['vertical_rate'])
    print(lista[aircraft]['sensors'])
    print(lista[aircraft]['baro_altitude'])
    print(lista[aircraft]['squawk'])
    print(lista[aircraft]['spi'])
    print(lista[aircraft]['position_source'])
    print(lista[aircraft]['category'])
    
    
aircraft_spark_df = spark.createDataFrame(aircraft_df, schema=schema)

ICAO24: 39de4f
TVF783R 
France
1786786986
1786786986
0.4812
45.7351
11833.86
False
221.18
214.58
0
None
11285.22
7630
False
0
0
ICAO24: 39de4e
TVF37NP 
France
1786786859
1786786980
25.1545
36.5105
2987.04
False
145.17
113.39
-7.15
None
2895.6
0652
False
0
0
ICAO24: 80162c
AXB386  
India
1786786985
1786786985
55.437
24.1562
12100.56
False
229.39
115.93
0
None
11277.6
3110
False
0
0
ICAO24: a09281
N136LM  
United States
1786786986
1786786986
-84.9257
33.8921
647.7
False
40.38
44.48
-0.98
None
579.12
None
False
0
0
ICAO24: 4031ae
GBSED   
United Kingdom
1786786986
1786786986
-0.8286
53.2354
746.76
False
49.66
76.83
1.3
None
670.56
7000
False
0
0
ICAO24: ac494e
CMD3    
United States
1786786986
1786786986
-122.016
37.9131
1524
False
57.19
183.09
0
None
1485.9
4261
False
0
0
ICAO24: 39de4a
TVF8098 
France
1786786742
1786786742
9.7938
36.3738
12207.24
False
248.48
155.93
-9.75
None
11010.9
7673
False
0
0
ICAO24: 39de4d
TVF44UV 
France
1786786984
1786786984
26.7629
36.205
8663.94
False
219.27

In [14]:
# aircraft_spark_df.show()

In [15]:
# aircraft_spark_df.explain(True)

In [ ]:

aircraft_spark_df = aircraft_spark_df.withColumn("ingestion_timestamp", current_timestamp())

In [10]:
jdbc_url = "jdbc:postgresql://localhost:5432/OpenSky"

In [11]:
target_table = 'OpenSky_Aircraft_Bronze'

In [12]:
import json
with open("db.json", "r") as f:
    connection_properties = json.load(f)

aircraft_spark_df.write \
.mode("append") \
.jdbc(url=jdbc_url, table=target_table, properties=connection_properties)